Análise de E-Commerce - Dataset: Olist

Neste projeto exploro um dataset real de e-commerce brasileiro para responder perguntas de negócio usando Python e SQL.

As perguntas são:

    - Quantos pedidos foram feitos e qual foi a receita total?
    - Qual a média de avaliação dos clientes?
    - Quais são os meses com mais vendas?   
    - Quais estados compram mais?
    - Qual a taxa de atraso  nas entregas? 


1. Importando as bibliotecas

In [1]:
Import pandas as pd 
Import sqlite3

print("Bibliotecas importadas com sucesso")

SyntaxError: invalid syntax (1654480374.py, line 1)

2. Carregando dados 
    - Apenas tabelas que serão usadas nesta análise

In [ ]:
orders    = pd.read_csv("data/raw/olist_orders_dataset.csv")
items     = pd.read_csv("data/raw/olist_order_items_dataset.csv")
reviews   = pd.read_csv("data/raw/olist_order_reviews_dataset.csv")
customers = pd.read_csv("data/raw/olist_customers_dataset.csv")

print(f"Pedidos:    {len(orders):,} linhas")
print(f"Itens:      {len(items):,} linhas")
print(f"Avaliações: {len(reviews):,} linhas")
print(f"Clientes:   {len(customers):,} linhas")


3. Entendendo o contúdo de cada tabela

In [ ]:
# Exibindo as primeiras linhas do DataFrame de pedidos
orders.head()

In [ ]:
# Exibindo um resumo do DataFrame de pedidos
orders.info()

In [ ]:
# Contando quantas vezes cada status de pedido aparece
orders['order_status']value_counts()

4. Limpeza dos dados
    - Vou converter as datas, transformar as strings de data em objetos datatime e criar colunas úteis para a análise.

In [ ]:
# Converter colunas de data para o formato datetime
orders['order_puchase_timestamp'] = pd.to_datetime(orders['order_puchase_timestamp'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])

# Criar coluna de ano/mes para análise temporal
orders['ano_mes'] = orders['order_purchase_timestamp'].dt.to_period('M').astype(str)

# Calcular dias de entrega
orders['dias_entrega'] = (orders['order_delivered_customer_date'] - orders['order_puchase_timestamp']).dt.days

# Marcar entregas atrasadas: 1 = atrasada, 0 = no prazo
orders['entrega_atrasada'] = (orders['order_delivered_customer_date'] > orders['order_estimated_delivery_date']).astype(int)

print("Análise inicial concluída com sucesso")
print(f"Tempo médio de entrega: {orders['dias_entrega'].mean():.1f} dias")


In [ ]:
# calcular valor total de cada item (produto + frete)
items['valor_total'] = items['price'] + items['freight_value']

print(f"receita total: R$ {items['valor_total'].sum():,.2f}")
print(f"ticket médio por item: R$ {items['price'].mean():,.2f}")

5. Análise das principais métricas de negócio (KPIs)


In [ ]:
# Filtrar só pedidos entregues
entregues = orders[orders['order_status'] == 'delivered']

# Juntar pedidos com itens para calcular receita
pedidos_com_valor = entregues.merge(items, on= 'order_id', how='left')

# calcular KPIs
total_pedidos = entregues['order_id'].nunique()
receita_total = pedidos_com_valor['valor_total'].sum()
ticket_medio = pedidos_com_valor.groupby('order_id')['valor_total'].sum().mean()
nota_media = reviews['review_score'].mean() * 100
taxa_atraso = entregues['entrega_atrasada'].mean() * 100

print("=" * 40)
print(" KPIs DO NEGÓCIO")
print("=" * 40)
print(f" Total de pedidos entregues : {total_pedidos:,}")
print(f" Receita total              : R$ {receita_total:,.2f}")
print(f" Ticket médio por pedido    : R$ {ticket_medio:.2f}")
print(f" Nota média dos clientes    : {nota_media:.2f} / 5.0")
print(f" Taxa de atraso             : {taxa_atraso:.1f}%")
print("=" * 40)

6. Criando o Banco de dados
    - Irei salvar os dados limpos em um banco de dados para fazer consultas em SQL.

In [ ]:
conn = sqlite3.connect('olist.db')

orders.to_sql('orders', conn, if_exists='replace', index=false)
items.to_sql('items', conn, if_exists='replace', index=false)
reviews.to_sql('reviews', conn, if_exists='replace', index=false)
customers.to_sql('customers', conn, if_exists='replace', index=false)

print("Dados exportados para o banco de dados SQLite com sucesso")

7. Consultas SQL

Pergunta 1: Quais meses tiveram mais vendas?